# `sabench` Demonstration Notebook

**Variance-based (Sobol) global sensitivity analysis: benchmarks, transforms, and non-commutativity**

This notebook walks through three worked examples covering the core capabilities of `sabench`:

| Example | Benchmark | Output shape | Transform family |
|---------|-----------|-------------|-----------------|
| 1 — Scalar | Ishigami | `(N,)` | Pointwise mathematical |
| 2 — Temporal | SIR epidemic | `(N, T)` | Temporal aggregation |
| 3 — Spatial | Campbell 2D | `(N, 64, 64)` | Spatial aggregation |

## Setup

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json, pathlib, warnings
warnings.filterwarnings("ignore")

from sabench import (
    Ishigami, EpidemicSIR, Campbell2D,
    saltelli_sample, jansen_s1_st,
    apply_transform, score_transform,
    TRANSFORMS, POINTWISE_TRANSFORMS, MONOTONE_TRANSFORMS,
    CONVEX_TRANSFORMS, CONCAVE_TRANSFORMS, SMOOTH_TRANSFORMS,
)

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11,
})
print(f"sabench loaded. {len(TRANSFORMS)} transforms registered.")

---
## Example 1 — Scalar: Ishigami function

$$f(x_1,x_2,x_3) = \sin(x_1) + a\sin^2(x_2) + b\,x_3^4\sin(x_1), \quad x_i\in[-\pi,\pi]$$

Analytical S₁ (a=7, b=0.1): **[0.314, 0.442, 0.000]**

In [ ]:
model = Ishigami()
N = 2048
X = saltelli_sample(model.d, model.bounds, N=N, seed=42)
Y = model.evaluate(X)
S1, ST = jansen_s1_st(Y, N=N, d=model.d)
S1c = np.clip(S1, 0, 1);  STc = np.clip(ST, 0, 1)
S1_ana = model.analytical_S1()
print(f"Y shape: {Y.shape}  range [{Y.min():.2f}, {Y.max():.2f}]")
print(f"Analytical S1: {S1_ana.round(3)}")
print(f"Estimated  S1: {S1c.round(3)}")

In [ ]:
scalar_transforms = [
    ("affine_a2_b1",     "Affine (a=2,b=1)"),
    ("tanh_a03",         "Tanh (α=0.3)"),
    ("square_pointwise", "y² (even convex)"),
    ("log1p_positive",   "log(1+|y|) (concave)"),
    ("sin_pointwise",    "sin(fy) (periodic)"),
    ("step_pointwise",   "Heaviside step"),
    ("exp_pointwise",    "exp(0.1y) (convex)"),
    ("soft_threshold",   "Soft threshold"),
    ("swish",            "Swish activation"),
    ("hermite_he3",      "Hermite He3"),
]

s_results = []
for key, label in scalar_transforms:
    Y_t = apply_transform(Y, key)
    S1_t, _ = jansen_s1_st(Y_t, N=N, d=model.d)
    S1_tc = np.clip(S1_t, 0, 1)
    sc = score_transform(S1c, S1_tc, Y, Y_t)
    s_results.append({"key": key, "label": label, "S1_t": S1_tc,
                      "D": sc["D"], "delta": sc["delta"], "tf": sc["threshold_flip"]})

print(f"{'Label':<22}  {'S1_trans':>20}  {'D':>6}  {'Delta':>6}  {'TF':>3}")
print("─"*62)
for r in s_results:
    s = "  ".join(f"{float(v):.3f}" for v in r["S1_t"])
    print(f"{r['label']:<22}  [{s}]  {r['D']:.4f}  {r['delta']:.4f}  {r['tf']:>3}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel A: original S1 vs analytical
ax = axes[0]
x = np.arange(model.d)
ax.bar(x - 0.2, S1_ana, 0.38, label="Analytical", color="#3B82F6", alpha=0.85)
ax.bar(x + 0.2, S1c,    0.38, label="Estimated",  color="#F97316", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels([f"$X_{i+1}$" for i in x])
ax.set_ylim(0, 0.65); ax.set_ylabel("First-order index $S_1$")
ax.set_title("(A) Ishigami: original S1"); ax.legend(frameon=False)

# Panel B: D vs Delta scatter
ax = axes[1]
Dv = [r["D"] for r in s_results]; dv = [r["delta"] for r in s_results]
ax.scatter(Dv, dv, c=range(len(s_results)), cmap="tab10", s=80, zorder=3)
for i, r in enumerate(s_results):
    ax.annotate(r["label"], (Dv[i], dv[i]), fontsize=7,
                xytext=(4,3), textcoords="offset points")
ax.set_xlabel("Decision Score D"); ax.set_ylabel("Sensitivity Shift Delta")
ax.set_title("(B) Non-commutativity scores")

# Panel C: S1 heatmap
ax = axes[2]
mat = np.vstack([S1c] + [r["S1_t"] for r in s_results])
im = ax.imshow(mat, aspect="auto", vmin=0, vmax=0.6, cmap="RdYlGn_r")
ax.set_xticks(range(model.d)); ax.set_xticklabels([f"X{i+1}" for i in range(model.d)])
lbls = ["orig"] + [r["label"][:14] for r in s_results]
ax.set_yticks(range(len(lbls))); ax.set_yticklabels(lbls, fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.04); ax.set_title("(C) S1 under transforms")

fig.suptitle("Ishigami — Sobol index non-commutativity", fontsize=12)
fig.tight_layout()
plt.savefig("notebooks/scalar_demo.png", bbox_inches="tight", dpi=150)
plt.show(); print("Saved scalar_demo.png")

---
## Example 2 — Temporal: SIR epidemic model

ODE compartment model $\dot{S}=-\beta SI$, $\dot{I}=\beta SI-\gamma I$,
$\dot{R}=\gamma I$ with d=3 uncertain inputs (β, γ, I₀).
Output: infectious trajectory $I(t)$ over T=100 time steps.

In [ ]:
sir = EpidemicSIR()
N_sir = 512
X_sir = saltelli_sample(sir.d, sir.bounds, N=N_sir, seed=7)
Y_sir = sir.evaluate(X_sir)
print(f"SIR: d={sir.d}  Y shape: {Y_sir.shape}  range [{Y_sir.min():.3f}, {Y_sir.max():.3f}]")
S1_sir, ST_sir = jansen_s1_st(Y_sir, N=N_sir, d=sir.d)
S1_sir = np.clip(S1_sir, 0, 1)
print(f"S1 (raw trajectory): {S1_sir.round(3)}")

In [ ]:
temporal_transforms = [
    ("temporal_peak",                "Peak I(t)"),
    ("temporal_cumsum",              "Cumsum (linear)"),
    ("temporal_log_cumsum",          "Log-cumsum"),
    ("temporal_rms",                 "RMS"),
    ("temporal_quantile_q90",        "Q90"),
    ("temporal_exceedance_duration", "Exceedance dur."),
    ("temporal_envelope",            "Running max"),
    ("temporal_range",               "Range max-min"),
]

def vw_s1(S1_2d, Y_2d):
    """Variance-weighted aggregate S1: (d, T) -> (d,)."""
    var_t = Y_2d.var(axis=0)
    tv = var_t.sum()
    if tv > 1e-12:
        return np.clip((S1_2d * var_t[None, :]).sum(axis=1) / tv, 0, 1)
    return np.clip(S1_2d.mean(axis=1), 0, 1)

sir_results = []
for key, label in temporal_transforms:
    Y_t = apply_transform(Y_sir, key)
    S1_t, _ = jansen_s1_st(Y_t, N=N_sir, d=sir.d)
    S1_tc = np.clip(S1_t, 0, 1)          # (d, T)
    S1_agg = vw_s1(S1_tc, Y_t)           # (d,) for display
    sc = score_transform(S1_sir, S1_tc, Y_sir, Y_t)
    sir_results.append({"key": key, "label": label, "S1_t": S1_agg,
                        "D": sc["D"], "delta": sc["delta"]})

print(f"{'Transform':<26}  {'S1_trans (agg)':>20}  {'D':>6}  {'Delta':>6}")
print('\u2500'*64)
for r in sir_results:
    s = "  ".join(f"{float(v):.3f}" for v in r["S1_t"])
    print(f"{r['label']:<26}  [{s}]  {r['D']:.4f}  {r['delta']:.4f}")


In [ ]:
n_in = sir.d
xlabels = [f"X{i+1}" for i in range(n_in)]
colors = plt.cm.tab10(np.linspace(0, 0.85, len(sir_results)))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: sample trajectories
ax = axes[0]
for i in range(min(40, len(Y_sir))):
    ax.plot(Y_sir[i], color="steelblue", alpha=0.2, lw=0.7)
ax.plot(Y_sir[0], color="navy", lw=1.8)
ax.set_xlabel("Time step"); ax.set_ylabel("Infectious I(t)")
ax.set_title("(A) SIR trajectories (40 samples)")

# Panel B: S1 bar chart per transform
ax = axes[1]
n_t = len(sir_results)
x = np.arange(n_in); w = 0.7 / n_t
for j, r in enumerate(sir_results):
    off = (j - n_t/2) * w + w/2
    ax.bar(x + off, r["S1_t"], w * 0.9, color=colors[j],
           label=r["label"], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(xlabels)
ax.set_ylabel("S1"); ax.set_title("(B) S1 by temporal transform")
ax.legend(fontsize=7, ncol=2, frameon=False)

# Panel C: D vs Delta
ax = axes[2]
for j, r in enumerate(sir_results):
    ax.scatter(r["D"], r["delta"], color=colors[j], s=90, zorder=3)
    ax.annotate(r["label"], (r["D"], r["delta"]), fontsize=7,
                xytext=(4,3), textcoords="offset points")
ax.set_xlabel("Decision Score D"); ax.set_ylabel("Sensitivity Shift Delta")
ax.set_title("(C) Non-commutativity")

fig.suptitle("SIR epidemic — temporal aggregation transforms", fontsize=12)
fig.tight_layout()
plt.savefig("notebooks/temporal_demo.png", bbox_inches="tight", dpi=150)
plt.show(); print("Saved temporal_demo.png")

---
## Example 3 — Spatial: Campbell 2D random field

Each sample produces a 64×64 random field driven by 5 uncertain parameters
(2 amplitudes, 3 rate parameters). We compare spatial aggregation transforms.

In [ ]:
cam = Campbell2D(n_z=32)          # 32x32 field
N_cam = 256
X_cam = saltelli_sample(cam.d, cam.bounds, N=N_cam, seed=13)
Y_cam = cam.evaluate(X_cam)
n_total_cam = Y_cam.shape[0]
print(f"Campbell2D: d={cam.d}  Y shape: {Y_cam.shape}")

# For SA on spatial fields: flatten to (n_total, H*W), then variance-weight
def s1_spatial(Y, N, d):
    n = Y.shape[0]
    Yf = Y.reshape(n, -1)
    S1_px, ST_px = jansen_s1_st(Yf, N=N, d=d)   # (d, H*W)
    var_px = Yf.var(axis=0);  tv = var_px.sum()
    wS1 = (S1_px * var_px[None,:]).sum(1) / max(tv, 1e-12) if tv > 1e-12 else S1_px.mean(1)
    return np.clip(wS1, 0, 1), np.clip(S1_px, 0, 1), Yf

S1_cam, S1_px_cam, Y_cam_flat = s1_spatial(Y_cam, N_cam, cam.d)
print(f"S1 (var-weighted): {S1_cam.round(3)}")

In [ ]:
spatial_transforms = [
    ("regional_mean",       "Regional mean"),
    ("block_2x2",           "Block 2x2"),
    ("block_4x4",           "Block 4x4"),
    ("exceedance_q75",      "Exceedance q75"),
    ("exceedance_area",     "Exceed. area"),
    ("matern_smooth",       "Matern smooth"),
    ("laplacian_roughness", "Laplacian"),
    ("gradient_magnitude",  "Gradient mag."),
    ("log_shift",           "Log-shift"),
    ("tanh_a03",            "Tanh (a=0.3)"),
]

cam_results = []
for key, label in spatial_transforms:
    Y_t = apply_transform(Y_cam, key)
    S1_t, _, Y_t_flat = s1_spatial(Y_t, N_cam, cam.d)
    sc = score_transform(S1_cam, S1_t, Y_cam_flat, Y_t_flat)
    cam_results.append({"key": key, "label": label, "S1_t": S1_t,
                        "D": sc["D"], "delta": sc["delta"]})

print(f"{'Transform':<20}  {'S1_trans':>34}  {'D':>6}  {'Delta':>6}")
print("-"*72)
for r in cam_results:
    s = "  ".join(f"{float(v):.3f}" for v in r["S1_t"])
    print(f"{r['label']:<20}  [{s}]  {r['D']:.4f}  {r['delta']:.4f}")

In [ ]:
n_in = cam.d; xlabels = [f"X{i+1}" for i in range(n_in)]
colors = plt.cm.tab20(np.linspace(0, 0.85, len(cam_results)))
HW = int(Y_cam.shape[1])

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# Panel A: example field
ax0 = fig.add_subplot(gs[0, 0])
im0 = ax0.imshow(Y_cam[0], origin="lower", cmap="viridis")
plt.colorbar(im0, ax=ax0, fraction=0.05)
ax0.set_title("(A) Example field (sample 1)")

# Panel B: mean field
ax1 = fig.add_subplot(gs[0, 1])
im1 = ax1.imshow(Y_cam[:50].mean(0), origin="lower", cmap="viridis")
plt.colorbar(im1, ax=ax1, fraction=0.05)
ax1.set_title("(B) Mean field (50 samples)")

# Panel C: per-pixel S1 for dominant input
ax2 = fig.add_subplot(gs[0, 2])
dom = int(np.argmax(S1_cam))
S1_field = S1_px_cam[dom].reshape(HW, HW)
im2 = ax2.imshow(S1_field, origin="lower", cmap="YlOrRd", vmin=0, vmax=0.8)
plt.colorbar(im2, ax=ax2, fraction=0.05)
ax2.set_title(f"(C) Per-pixel S1, X{dom+1} (dominant)")

# Panel D: S1 grouped bar
ax3 = fig.add_subplot(gs[1, 0:2])
n_t = len(cam_results); x = np.arange(n_in); w = 0.72 / n_t
for j, r in enumerate(cam_results):
    off = (j - n_t/2) * w + w/2
    ax3.bar(x + off, r["S1_t"], w * 0.9, color=colors[j], label=r["label"], alpha=0.85)
ax3.set_xticks(x); ax3.set_xticklabels(xlabels)
ax3.set_ylabel("S1"); ax3.set_title("(D) S1 by spatial transform")
ax3.legend(fontsize=7, ncol=5, frameon=False,
           bbox_to_anchor=(0.5, -0.22), loc="upper center")

# Panel E: D vs Delta
ax4 = fig.add_subplot(gs[1, 2])
for j, r in enumerate(cam_results):
    ax4.scatter(r["D"], r["delta"], color=colors[j], s=80, zorder=3)
    ax4.annotate(r["label"], (r["D"], r["delta"]),
                 fontsize=7, xytext=(4,2), textcoords="offset points")
ax4.set_xlabel("Decision Score D"); ax4.set_ylabel("Sensitivity Shift Delta")
ax4.set_title("(E) Non-commutativity")

fig.suptitle("Campbell 2D — spatial transforms", fontsize=12)
plt.savefig("notebooks/spatial_demo.png", bbox_inches="tight", dpi=150)
plt.show(); print("Saved spatial_demo.png")

---
## Bonus: Transform catalogue overview

In [ ]:
meta_path = pathlib.Path("sabench/metadata/transforms_metadata.json")
with open(meta_path) as f:
    tmeta = json.load(f)

from collections import Counter
cats = dict(sorted(Counter(v["category"] for v in tmeta.values()).items(),
                   key=lambda x: -x[1]))
props = {
    "Pointwise (74)":  len(POINTWISE_TRANSFORMS),
    "Monotone (56)":   len(MONOTONE_TRANSFORMS),
    "Concave (30)":    len(CONCAVE_TRANSFORMS),
    "Convex (23)":     len(CONVEX_TRANSFORMS),
    "Smooth C-inf (90)": len(SMOOTH_TRANSFORMS),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
bars = ax.barh(list(cats.keys()), list(cats.values()),
               color=plt.cm.Set2(np.linspace(0, 1, len(cats))))
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_xlabel("Number of transforms")
ax.set_title(f"(A) 172 transforms by category")

ax = axes[1]
bars2 = ax.bar(list(props.keys()), list(props.values()),
               color=["#3B82F6","#10B981","#EF4444","#F59E0B","#8B5CF6"], alpha=0.85)
ax.bar_label(bars2, padding=3, fontsize=9)
ax.set_ylabel("Count"); ax.set_title("(B) Transforms by property")
ax.tick_params(axis="x", rotation=15)

fig.suptitle("sabench — 172 output transform catalogue", fontsize=12)
fig.tight_layout()
plt.savefig("notebooks/transform_landscape.png", bbox_inches="tight", dpi=150)
plt.show(); print("Saved transform_landscape.png")

---
## Summary

| | Ishigami (scalar) | SIR (temporal) | Campbell 2D (spatial) |
|--|--|--|--|
| Output shape | `(N,)` | `(N, 100)` | `(N, 32, 32)` |
| d (inputs) | 3 | 3 | 5 |
| Most disruptive transform | `step_pointwise` | `temporal_peak` | `laplacian_roughness` |

**Key observations:**
- **Affine transforms** leave S₁ exactly unchanged — D ≈ 0, Δ ≈ 0 (positive control).
- **Threshold/step transforms** produce the largest non-commutativity in scalar output.
- **Temporal peak extraction** amplifies the input controlling the infectious peak.
- **Spatial block averaging** smooths index differences as block size grows.
- **Laplacian roughness** can invert dominant input ordering by amplifying local gradients.